# OBD Baseline Evaluation

This notebook computes baseline results for the OBD task on the **same test split** used by the training notebooks.

What this notebook does:

1. Loads `data/obd_cot_gpt5.jsonl`
2. Rebuilds the no-leak prompt exactly as in the OBD notebooks
3. Recreates the same `train/val/test` split with `SEED = 42`
4. Evaluates baseline models on the **test set only**
5. Saves predictions, detailed metrics, and summary files

This baseline notebook is intentionally focused on the OBD workflow so the repo can stay lean.

In [8]:
from __future__ import annotations

import json
import os
import random
import re
import time
from collections import Counter
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any
from urllib import request

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from opentslm.system_metrics import SystemMetricsMonitor

try:
    from transformers import pipeline
except ImportError:
    pipeline = None


## Configuration

Choose the backend and the baseline models you want to run. The default setup is text-only baseline inference on the OBD test set.

In [9]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT.parent / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / 'data' / 'obd_cot_gpt5.jsonl'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'obd_baselines'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR = PROJECT_ROOT / 'results' / 'notebook_metrics'
METRICS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

BACKEND = 'hf'  # 'hf', 'openai', or 'ollama'
MODEL_NAMES = [
    'google/gemma-3-270m',
    'meta-llama/Llama-3.2-1B',
]

MAX_SAMPLES = None  # set an int for a quick dry run
MAX_NEW_TOKENS = 220
TEMPERATURE = 0.1
SAVE_EVERY = 10
SLEEP_BETWEEN_CALLS = 0.0
REQUEST_TIMEOUT = 600

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '')
OPENAI_BASE_URL = os.getenv('OPENAI_BASE_URL', 'https://api.openai.com/v1/chat/completions')
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')

FEATURE_LABELS = ['Speed', 'RPM', 'Engine Load', 'Throttle Position']
LABELS = ['economical', 'normal', 'aggressive']

_NO_LEAK_PROMPT = """You are shown a time-series plot of vehicle telemetry over a 120-sample window.
This data corresponds to one of two possible activities:
[{label}]
[{dis}]
Your task is to classify the activity based on analysis of the data.
Instructions:
- Begin by analyzing the time-series without assuming a specific label.
- Think step-by-step about what the observed patterns suggest regarding movement intensity and behavior.
- Write your rationale as a single, natural paragraph, do not use bullet points, numbered steps, or section headings.
- Do not refer back to the plot or to the act of visual analysis in your rationale; the plot is only for reference but you should reason about the time-series data.
- Do not assume any answer at the beginning, analyze as if you do not yet know which class is correct.
- Do not mention either class label until the final sentence.
- Make sure that your last word is the answer. You MUST end your response with "Answer:"."""

print('Data path:', DATA_PATH)
print('Output dir:', OUTPUT_DIR)
print('Metrics dir:', METRICS_DIR)
print('Backend:', BACKEND)
print('Models:', MODEL_NAMES)


Data path: c:\Users\Morsinaldo\Desktop\OpenTSLM\data\obd_cot_gpt5.jsonl
Output dir: c:\Users\Morsinaldo\Desktop\OpenTSLM\data\obd_baselines
Backend: hf
Models: ['google/gemma-3-270m', 'meta-llama/Llama-3.2-1B']


## Load Data and Recreate the Same Test Split

In [10]:
rows = [json.loads(x) for x in DATA_PATH.read_text(encoding='utf-8').splitlines() if x.strip()]
rows = [r for r in rows if r.get('label') != 'congested']


def build_no_leak_prompt(row: dict[str, Any]) -> str:
    return _NO_LEAK_PROMPT.format(label=row.get('label') or '', dis=row.get('dissimilar_label') or '')


for row in rows:
    row['pre_prompt'] = build_no_leak_prompt(row)

idx = list(range(len(rows)))
random.shuffle(idx)

n = len(idx)
train_end = max(1, int(0.6 * n))
val_end = max(train_end + 1, int(0.8 * n)) if n > 2 else train_end

train_rows = [rows[i] for i in idx[:train_end]]
val_rows = [rows[i] for i in idx[train_end:val_end]]
test_rows = [rows[i] for i in idx[val_end:]]

if len(val_rows) == 0 and len(train_rows) > 1:
    val_rows = [train_rows.pop()]
if len(test_rows) == 0 and len(train_rows) > 1:
    test_rows = [train_rows.pop()]

if MAX_SAMPLES is not None:
    test_rows = test_rows[:MAX_SAMPLES]

print('train/val/test:', len(train_rows), len(val_rows), len(test_rows))
print('Example test id:', test_rows[0]['id'] if test_rows else 'N/A')


train/val/test: 115 39 39
Example test id: obd_cot_000088


## Build the Text-Only OBD Input

This follows the same textual signal summarization used in the OBD soft-prompt notebook.

In [11]:
def summarize_series(label: str, values: torch.Tensor, n_points: int = 8) -> str:
    values = values.detach().cpu().float()
    n = values.numel()
    idx = torch.linspace(0, max(n - 1, 0), steps=min(n_points, n)).long()
    samples = ', '.join(f'{values[i].item():.1f}' for i in idx)
    delta = values[-1].item() - values[0].item()
    if delta > 1e-3:
        trend = 'overall increasing'
    elif delta < -1e-3:
        trend = 'overall decreasing'
    else:
        trend = 'roughly flat'
    return (
        f'{label} series with mean {values.mean().item():.2f}, '
        f'std {values.std(unbiased=False).item():.2f}, '
        f'min {values.min().item():.2f}, max {values.max().item():.2f}, '
        f'trend {trend}, sampled points [{samples}]:'
    )


def build_input_text(row: dict[str, Any]) -> str:
    ts = torch.tensor(row['time_series'], dtype=torch.float32)
    if ts.ndim != 2 or ts.shape[0] < 4:
        raise ValueError(f"Unexpected time_series shape for id={row.get('id')}: {tuple(ts.shape)}")
    summaries = [
        summarize_series(label, series)
        for label, series in zip(FEATURE_LABELS, [ts[0], ts[1], ts[2], ts[3]])
    ]
    return f"{row['pre_prompt']} | {' | '.join(summaries)} | {row.get('post_prompt', 'Rationale:')}"


test_examples = [
    {
        'id': row['id'],
        'input_text': build_input_text(row),
        'target': row['answer'],
        'reference_label': row.get('label'),
    }
    for row in test_rows
]

print('Built test examples:', len(test_examples))
print(test_examples[0]['input_text'][:800] if test_examples else 'No test examples')


Built test examples: 39
You are shown a time-series plot of vehicle telemetry over a 120-sample window.
This data corresponds to one of two possible activities:
[normal]
[aggressive]
Your task is to classify the activity based on analysis of the data.
Instructions:
- Begin by analyzing the time-series without assuming a specific label.
- Think step-by-step about what the observed patterns suggest regarding movement intensity and behavior.
- Write your rationale as a single, natural paragraph, do not use bullet points, numbered steps, or section headings.
- Do not refer back to the plot or to the act of visual analysis in your rationale; the plot is only for reference but you should reason about the time-series data.
- Do not assume any answer at the beginning, analyze as if you do not yet know which class is corr


## Inference Helpers

In [12]:
def model_suffix(model_name: str) -> str:
    suffix = model_name.split('/')[-1].replace(':', '_').replace('-', '_').replace('.', '_')
    return suffix.lower()


def post_json(url: str, payload: dict[str, Any], headers: dict[str, str] | None = None) -> dict[str, Any]:
    body = json.dumps(payload).encode('utf-8')
    req = request.Request(url, data=body, headers={**(headers or {}), 'Content-Type': 'application/json'})
    with request.urlopen(req, timeout=REQUEST_TIMEOUT) as response:
        return json.loads(response.read().decode('utf-8'))


def generate_with_openai(model_name: str, prompt: str) -> str:
    if not OPENAI_API_KEY:
        raise ValueError('OPENAI_API_KEY is not set.')
    payload = {
        'model': model_name,
        'temperature': TEMPERATURE,
        'messages': [{'role': 'user', 'content': prompt}],
        'max_tokens': MAX_NEW_TOKENS,
    }
    headers = {'Authorization': f'Bearer {OPENAI_API_KEY}'}
    data = post_json(OPENAI_BASE_URL, payload, headers=headers)
    return data['choices'][0]['message']['content'].strip()


def generate_with_ollama(model_name: str, prompt: str) -> str:
    payload = {
        'model': model_name,
        'stream': False,
        'messages': [{'role': 'user', 'content': prompt}],
        'options': {'temperature': TEMPERATURE, 'num_predict': MAX_NEW_TOKENS},
    }
    data = post_json(f'{OLLAMA_BASE_URL}/api/chat', payload)
    return data['message']['content'].strip()


def load_hf_pipeline(model_name: str):
    if pipeline is None:
        raise ImportError('transformers is not installed in this environment.')
    return pipeline(
        task='text-generation',
        model=model_name,
        torch_dtype='auto',
        device_map='auto',
        temperature=TEMPERATURE,
    )


def generate_with_hf(pipe, prompt: str) -> str:
    outputs = pipe(prompt, max_new_tokens=MAX_NEW_TOKENS, return_full_text=False)
    if not outputs:
        return ''
    return outputs[0]['generated_text'].strip()


## Metrics

These are the same core metrics already used in your OBD notebooks, so the baseline outputs stay directly comparable.

In [13]:
def normalize_tokens(text: str) -> list[str]:
    return re.findall(r'[a-z0-9_\-]+', str(text).lower())


def token_f1(pred: str, gold: str) -> float:
    p = normalize_tokens(pred)
    g = normalize_tokens(gold)
    if not p or not g:
        return 0.0
    common = Counter(p) & Counter(g)
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(p)
    recall = overlap / len(g)
    return 2 * precision * recall / (precision + recall)


def rouge_l_f1(pred: str, gold: str) -> float:
    p = normalize_tokens(pred)
    g = normalize_tokens(gold)
    if not p or not g:
        return 0.0
    dp = [[0] * (len(g) + 1) for _ in range(len(p) + 1)]
    for i in range(1, len(p) + 1):
        for j in range(1, len(g) + 1):
            if p[i - 1] == g[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    lcs = dp[-1][-1]
    if lcs == 0:
        return 0.0
    precision = lcs / len(p)
    recall = lcs / len(g)
    return 2 * precision * recall / (precision + recall)


def exact_match(pred: str, gold: str) -> int:
    return int(str(pred).strip().lower() == str(gold).strip().lower())


def seq_sim(pred: str, gold: str) -> float:
    return SequenceMatcher(None, str(pred).strip().lower(), str(gold).strip().lower()).ratio()


def extract_label(text: str | None) -> str | None:
    if not text:
        return None
    t = str(text).strip().lower()
    m = re.findall(r'answer\s*:\s*([a-z_\-]+)', t)
    if m:
        return m[-1]
    for label in LABELS:
        if re.search(rf'{re.escape(label)}', t):
            return label
    return None


def classification_metrics(predictions: list[dict[str, Any]]) -> tuple[float, float, list[dict[str, Any]]]:
    rows = []
    correct = 0
    total = 0
    for item in predictions:
        gt = extract_label(item.get('target'))
        pr = extract_label(item.get('prediction'))
        is_correct = int(gt is not None and pr == gt)
        correct += is_correct
        total += 1
        rows.append({'id': item['id'], 'gt_label': gt, 'pred_label': pr, 'accuracy': is_correct})

    accuracy = (correct / total) if total else 0.0

    macro_f1_sum = 0.0
    for label in LABELS:
        tp = sum(1 for r in rows if r['gt_label'] == label and r['pred_label'] == label)
        fp = sum(1 for r in rows if r['gt_label'] != label and r['pred_label'] == label)
        fn = sum(1 for r in rows if r['gt_label'] == label and r['pred_label'] != label)
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
        macro_f1_sum += f1
    macro_f1 = macro_f1_sum / len(LABELS) if LABELS else 0.0
    return accuracy, macro_f1, rows


def build_metrics(predictions: list[dict[str, Any]]) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    detailed_rows = []
    token_scores, rouge_scores, em_scores, seq_scores = [], [], [], []

    for item in predictions:
        tok = token_f1(item['prediction'], item['target'])
        rou = rouge_l_f1(item['prediction'], item['target'])
        em = exact_match(item['prediction'], item['target'])
        sim = seq_sim(item['prediction'], item['target'])
        pred_label = extract_label(item['prediction'])
        has_answer_tag = bool(re.search(r'answer\s*:', item['prediction'].lower()))
        non_ascii_ratio = (sum(ord(ch) > 127 for ch in item['prediction']) / max(len(item['prediction']), 1))

        row = dict(item)
        row.update({
            'token_f1': float(tok),
            'rouge_l_f1': float(rou),
            'exact_match': int(em),
            'seq_sim': float(sim),
            'pred_label': pred_label,
            'valid_label': pred_label in LABELS,
            'has_answer_tag': has_answer_tag,
            'prediction_length': len(item['prediction']),
            'non_ascii_ratio': float(non_ascii_ratio),
        })
        detailed_rows.append(row)
        token_scores.append(tok)
        rouge_scores.append(rou)
        em_scores.append(em)
        seq_scores.append(sim)

    accuracy, macro_f1, cls_rows = classification_metrics(predictions)
    cls_by_id = {r['id']: r for r in cls_rows}
    for row in detailed_rows:
        row.update(cls_by_id[row['id']])

    summary = {
        'n_samples': len(predictions),
        'token_f1_mean': float(np.mean(token_scores)) if token_scores else 0.0,
        'rouge_l_f1_mean': float(np.mean(rouge_scores)) if rouge_scores else 0.0,
        'exact_match_mean': float(np.mean(em_scores)) if em_scores else 0.0,
        'seq_sim_mean': float(np.mean(seq_scores)) if seq_scores else 0.0,
        'accuracy': float(accuracy),
        'macro_f1': float(macro_f1),
        'has_answer_tag_rate': float(np.mean([r['has_answer_tag'] for r in detailed_rows])) if detailed_rows else 0.0,
        'valid_label_rate': float(np.mean([r['valid_label'] for r in detailed_rows])) if detailed_rows else 0.0,
        'mean_prediction_length': float(np.mean([r['prediction_length'] for r in detailed_rows])) if detailed_rows else 0.0,
        'mean_non_ascii_ratio': float(np.mean([r['non_ascii_ratio'] for r in detailed_rows])) if detailed_rows else 0.0,
    }
    return detailed_rows, summary


## Run Baselines on the Test Set

This loop supports resume-by-file so you can stop and continue later without losing completed predictions.

In [14]:
all_model_summaries = []

for model_name in MODEL_NAMES:
    suffix = model_suffix(model_name)
    out_pred = OUTPUT_DIR / f'obd_baseline_predictions_{suffix}.jsonl'
    out_detail = OUTPUT_DIR / f'obd_baseline_detailed_{suffix}.jsonl'
    out_summary = OUTPUT_DIR / f'obd_baseline_summary_{suffix}.json'
    metrics_samples_path = METRICS_DIR / f'obd_baseline_{suffix}_samples.csv'
    metrics_summary_path = METRICS_DIR / f'obd_baseline_{suffix}_summary.json'

    predictions = []
    completed_ids = set()
    if out_pred.exists():
        with out_pred.open('r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                row = json.loads(line)
                predictions.append(row)
                completed_ids.add(row['id'])
        print(f'Resuming {model_name} from {len(predictions)} saved predictions.')

    pending_examples = [x for x in test_examples if x['id'] not in completed_ids]
    print(f'Running {model_name} on {len(pending_examples)} pending test examples...')

    model_monitor = SystemMetricsMonitor(
        label=f'obd_baseline_{suffix}',
        interval_s=0.5,
        disk_path=PROJECT_ROOT,
        metadata={'notebook': 'obd_baseline_evaluation', 'stage': 'inference', 'model_name': model_name, 'backend': BACKEND},
    ).start()
    model_monitor.mark('setup', pending_examples=len(pending_examples), resumed_examples=len(predictions))

    hf_pipe = None
    if BACKEND == 'hf' and pending_examples:
        pipeline_start = time.perf_counter()
        hf_pipe = load_hf_pipeline(model_name)
        model_monitor.mark('pipeline_loaded', pipeline_load_time_s=time.perf_counter() - pipeline_start)

    for idx, example in enumerate(tqdm(pending_examples, desc=f'{model_name}', unit='sample'), start=1):
        started = time.perf_counter()
        if BACKEND == 'hf':
            prediction = generate_with_hf(hf_pipe, example['input_text'])
        elif BACKEND == 'openai':
            prediction = generate_with_openai(model_name, example['input_text'])
        elif BACKEND == 'ollama':
            prediction = generate_with_ollama(model_name, example['input_text'])
        else:
            raise ValueError(f'Unsupported BACKEND: {BACKEND}')
        latency = time.perf_counter() - started

        predictions.append({
            'id': example['id'],
            'prompt': example['input_text'],
            'prediction': prediction,
            'target': example['target'],
            'model_name': model_name,
            'backend': BACKEND,
        })
        model_monitor.mark(
            'inference_step',
            step=idx,
            inference_latency_s=latency,
            prediction_chars=len(prediction),
            sample_id=example['id'],
        )

        if idx % SAVE_EVERY == 0 or idx == len(pending_examples):
            with out_pred.open('w', encoding='utf-8') as f:
                for row in predictions:
                    f.write(json.dumps(row, ensure_ascii=False) + '
')

        if SLEEP_BETWEEN_CALLS:
            time.sleep(SLEEP_BETWEEN_CALLS)

    detailed_rows, summary = build_metrics(predictions)
    summary.update({'model_name': model_name, 'backend': BACKEND})

    with out_detail.open('w', encoding='utf-8') as f:
        for row in detailed_rows:
            f.write(json.dumps(row, ensure_ascii=False) + '
')
    out_summary.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

    model_monitor.stop(final_phase='finished', n_predictions=len(predictions), pending_examples=len(pending_examples))
    model_monitor.to_csv(metrics_samples_path)
    metrics_summary_path.write_text(json.dumps(model_monitor.summary().to_dict(), ensure_ascii=False, indent=2), encoding='utf-8')

    all_model_summaries.append(summary)
    print(f'Saved predictions to {out_pred}')
    print(f'Saved detailed metrics to {out_detail}')
    print(f'Saved summary to {out_summary}')
    print(f'Saved metrics samples to {metrics_samples_path}')
    print(f'Saved metrics summary to {metrics_summary_path}')
    print(summary)


Running google/gemma-3-270m on 39 pending test examples...


`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0


google/gemma-3-270m:   0%|          | 0/39 [00:00<?, ?sample/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Saved predictions to c:\Users\Morsinaldo\Desktop\OpenTSLM\data\obd_baselines\obd_baseline_predictions_gemma_3_270m.jsonl
Saved detailed metrics to c:\Users\Morsinaldo\Desktop\OpenTSLM\data\obd_baselines\obd_baseline_detailed_gemma_3_270m.jsonl
Saved summary to c:\Users\Morsinaldo\Desktop\OpenTSLM\data\obd_baselines\obd_baseline_summary_gemma_3_270m.json
{'n_samples': 39, 'token_f1_mean': 0.08233550522688911, 'rouge_l_f1_mean': 0.07130861439688944, 'exact_match_mean': 0.0, 'seq_sim_mean': 0.019358174395912378, 'accuracy': 0.0, 'macro_f1': 0.0, 'has_answer_tag_rate': 0.0, 'valid_label_rate': 0.0, 'mean_prediction_length': 377.28205128205127, 'mean_non_ascii_ratio': 0.0, 'model_name': 'google/gemma-3-270m', 'backend': 'hf'}
Running meta-llama/Llama-3.2-1B on 39 pending test examples...


Device set to use cuda:0


meta-llama/Llama-3.2-1B:   0%|          | 0/39 [00:00<?, ?sample/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

Saved predictions to c:\Users\Morsinaldo\Desktop\OpenTSLM\data\obd_baselines\obd_baseline_predictions_llama_3_2_1b.jsonl
Saved detailed metrics to c:\Users\Morsinaldo\Desktop\OpenTSLM\data\obd_baselines\obd_baseline_detailed_llama_3_2_1b.jsonl
Saved summary to c:\Users\Morsinaldo\Desktop\OpenTSLM\data\obd_baselines\obd_baseline_summary_llama_3_2_1b.json
{'n_samples': 39, 'token_f1_mean': 0.15455142359901247, 'rouge_l_f1_mean': 0.12472438739075385, 'exact_match_mean': 0.0, 'seq_sim_mean': 0.018305904067688294, 'accuracy': 0.02564102564102564, 'macro_f1': 0.03333333333333333, 'has_answer_tag_rate': 0.02564102564102564, 'valid_label_rate': 0.02564102564102564, 'mean_prediction_length': 618.3589743589744, 'mean_non_ascii_ratio': 0.0, 'model_name': 'meta-llama/Llama-3.2-1B', 'backend': 'hf'}


## Compare Baseline Models

In [15]:
summary_df = pd.DataFrame(all_model_summaries).sort_values(['accuracy', 'macro_f1', 'token_f1_mean'], ascending=False)
display(summary_df)

comparison_path = OUTPUT_DIR / 'obd_baseline_model_comparison.csv'
summary_df.to_csv(comparison_path, index=False)
print('Saved comparison table to', comparison_path)


,n_samples,token_f1_mean,rouge_l_f1_mean,exact_match_mean,seq_sim_mean,accuracy,macro_f1,has_answer_tag_rate,valid_label_rate,mean_prediction_length,mean_non_ascii_ratio,model_name,backend
1,39,0.154551,0.124724,0.0,0.018306,0.025641,0.033333,0.025641,0.025641,618.358974,0.0,meta-llama/Llama-3.2-1B,hf
0,39,0.082336,0.071309,0.0,0.019358,0.000000,0.000000,0.000000,0.000000,377.282051,0.0,google/gemma-3-270m,hf


Saved comparison table to c:\Users\Morsinaldo\Desktop\OpenTSLM\data\obd_baselines\obd_baseline_model_comparison.csv


## Notes

- This notebook evaluates only the **test split**.
- It uses the same no-leak prompt reconstruction and the same random split protocol as the OBD training notebooks.
- The baseline here is a **text-only zero-shot / direct prompting baseline** over the summarized telemetry signals.
- If you want, the next cleanup step can be extracting the shared OBD helpers from the notebooks into one small reusable Python module.